# Task 2 — Chunk 2 (CPU only)
Set **Runtime > Change runtime type > Hardware accelerator: None**.
Run the cell, mount the same Drive, and upload `task2-chunk2.zip`.
This restores your verified Chunk 1 backup, installs and tests the continuation code, verifies or prepares pinned model files, and saves a fresh CPU-preparation snapshot. No assignment training or model-weight loading occurs.
Keep the final `preparation.json` path. GPU launch comes with the remaining evaluation/analysis jobs.

In [ ]:
"""Notebook launcher: CPU restore/install/check/download/save, with GPU disabled."""
from pathlib import Path
from uuid import uuid4
import subprocess
import sys
import zipfile

import torch
from google.colab import drive, files

if torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > Hardware accelerator: None first.")
drive.mount("/content/drive")
print("Upload task2-chunk2.zip (the small code package).", flush=True)
uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError("Upload exactly task2-chunk2.zip")
filename = next(iter(uploaded))
if not filename.endswith(".zip"):
    raise RuntimeError("Expected the Chunk 2 ZIP package")
package = Path("/content") / ("task2_chunk2_" + uuid4().hex[:8])
package.mkdir()
with zipfile.ZipFile(filename) as archive:
    names = set()
    for item in archive.infolist():
        target = (package / item.filename).resolve()
        if package.resolve() not in target.parents or item.filename in names or (item.external_attr >> 16) & 0o170000 == 0o120000:
            raise RuntimeError("Unsafe or duplicate package member")
        names.add(item.filename)
    archive.extractall(package)
if not (package / "setup_chunk2.py").is_file():
    raise RuntimeError("Expected Chunk 2 setup script at the ZIP root")
subprocess.run([sys.executable, str(package / "setup_chunk2.py")], check=True)
subprocess.run([sys.executable, "-m", "scripts.task2_cpu_prepare"],
               cwd="/content/ATML-PA2-Task2", check=True)
drive.flush_and_unmount()
print("CPU files saved and Drive flushed. Keep GPU off until the remaining jobs are ready.", flush=True)
